<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/GROK4dot1_XAI_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## XAIKEY

In [ ]:
!pip install xai-sdk -q

In [2]:
from xai_sdk import Client
from xai_sdk.chat import user, system

from google.colab import userdata
XAI_key = userdata.get('XAI_KEY')

# 1. Initialize the client
client = Client(
    api_host="api.x.ai",
    api_key=XAI_key
)


chat = client.chat.create(model="grok-4-1-fast-reasoning", temperature=0)

# Append system and user messages to the chat history
chat.append(system("You are a PhD-level mathematician."))
chat.append(user("Consider a $2025 \\times 2025$ grid of integers. Prove or disprove the following statement: If the sum of the integers in every $2 \\times 2$ subgrid is divisible by 4, then the sum of the integers in the entire $2025 \\times 2025$ grid is also divisible by 4. Provide a rigorous mathematical argument using techniques from linear algebra or modular arithmetic to support your conclusion."))

# Sample (generate) a response from the model
response = chat.sample()

# Print the generated response content
print(response.content)

### Disproof

The statement is false. We explicitly construct a $2025 \times 2025$ grid of integers satisfying the hypothesis (every $2 \times 2$ subgrid sums to a multiple of 4) but whose total sum is *not* a multiple of 4. All computations are performed in $\mathbb{Z}/4\mathbb{Z}$, and we work modulo 4 throughout.

#### Construction of the counterexample grid
Label the entries of the grid by $a_{i,j}$ for $i,j = 1, \dots, 2025$. Define the **first row** and **first column** as follows:
\[
a_{1,1} = 1, \quad a_{1,j} = 0 \text{ for } j = 2, \dots, 2025,
\]
\[
a_{i,1} = 0 \text{ for } i = 2, \dots, 2025.
\]
For all other entries $a_{i,j}$ with $i \geq 2$ and $j \geq 2$, define them **recursively** using the $2 \times 2$ condition via its bottom-right entry:
\[
a_{i,j} \equiv -a_{i-1,j-1} - a_{i-1,j} - a_{i,j-1} \pmod{4}, \quad i,j = 2, \dots, 2025.
\]
This uniquely determines the entire grid.

#### Verification that every $2 \times 2$ subgrid sums to $0 \pmod{4}$
Consider any $2 \times 

## CASE1

In [ ]:
# =============================================================================
#  GROK 4.1 + LangChain + LangGraph – Flight Search Demo
# =============================================================================

# ─── Install dependencies ───────────────────────────────────────────────────
!pip install -q langchain langchain-openai langchain-core langgraph langchain-classic

In [2]:
# ─── Imports ────────────────────────────────────────────────────────────────
import os
from typing import Annotated, List, TypedDict
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langgraph.graph import StateGraph, END

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ─── Configuration ──────────────────────────────────────────────────────────
if userdata:
    XAI_API_KEY = userdata.get('XAI_KEY')
else:
    XAI_API_KEY = os.environ.get('XAI_API_KEY')

if not XAI_API_KEY:
    raise ValueError("XAI_API_KEY not found. Set it in Colab secrets or as environment variable.")

# ─── LLM Setup ──────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model          = "grok-4-1-fast-reasoning",
    temperature    = 0.2,
    api_key        = XAI_API_KEY,
    base_url       = "https://api.x.ai/v1",
    max_retries    = 2,
    request_timeout= 90,
)

print(f"Connected to model: {llm.model}\n")

# ─── Tools ──────────────────────────────────────────────────────────────────
@tool
def flight_search_tool(origin: str, destination: str, date: str) -> List[dict] | dict:
    """Mock flight search – returns demo data only for London → Tokyo on 2026-08-15."""
    print(f"\n--- TOOL CALLED: flight_search_tool ({origin} → {destination} on {date}) ---")

    origin      = origin.lower().strip()
    destination = destination.lower().strip()

    if origin == "london" and destination == "tokyo" and date == "2026-08-15":
        return [
            {"flight_number": "BA007", "airline": "British Airways",   "price_gbp": 1200, "duration_h": 13, "layovers": 0},
            {"flight_number": "JAL404", "airline": "Japan Airlines",   "price_gbp": 1150, "duration_h": 14, "layovers": 1},
        ]
    return {"message": "No matching flights found (this is a mock tool)."}

tools = [flight_search_tool]

# ─── Flight Search Agent ────────────────────────────────────────────────────
search_system_prompt = """You are a helpful flight search assistant.
Use the flight_search_tool when the user asks about flights, prices or availability.
Clearly extract origin, destination and date from the query.
After getting results, provide a short summary and pass to the review step."""

search_prompt = ChatPromptTemplate.from_messages([
    ("system", search_system_prompt),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("agent_scratchpad"),
])

# ─── Corrected creation ─────────────────────────────────────────────────────
flight_search_runnable = create_tool_calling_agent(
    llm,                      # 1st arg: LLM (or llm.bind_tools(tools))
    tools,                    # 2nd arg: list of tools
    search_prompt,            # 3rd arg: prompt template
)

flight_search_executor = AgentExecutor(
    agent           = flight_search_runnable,
    tools           = tools,
    verbose         = True,
    handle_parsing_errors = True,
    max_iterations  = 6,
)

# ─── Review / Summary Agent ─────────────────────────────────────────────────
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a concise flight summary specialist.
Look at the full conversation history.
Create a clean, friendly markdown summary.
Highlight the cheapest flight if multiple options exist.
Use bullet points or short paragraphs. Offer further help at the end."""),
    MessagesPlaceholder("messages"),
])

review_chain = review_prompt | llm

# ─── LangGraph State & Nodes ────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

def run_flight_search(state: AgentState):
    print("\n─── Flight Search Agent ───")
    result = flight_search_executor.invoke({"messages": state["messages"]})
    output = result.get("output", "No result from search agent.")
    return {"messages": [AIMessage(content=output)]}

def run_review(state: AgentState):
    print("\n─── Review / Summary Agent ───")
    response = review_chain.invoke(state)
    return {"messages": [AIMessage(content=response.content)]}

# ─── Build Graph ────────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

workflow.add_node("search", run_flight_search)
workflow.add_node("review", run_review)

workflow.set_entry_point("search")
workflow.add_edge("search", "review")
workflow.add_edge("review", END)

app = workflow.compile()

print("Graph compiled successfully.\n")

# ─── Run Demo ───────────────────────────────────────────────────────────────
query = "Find me flights from London to Tokyo for August 15, 2026."

print("User query:", query, "\n")

initial_input = {"messages": [HumanMessage(content=query)]}

try:
    final_output = app.invoke(initial_input)

    print("\n" + "═"*70)
    print("FINAL ANSWER")
    print("═"*70 + "\n")

    print(final_output["messages"][-1].content)

except Exception as e:
    print(f"\nExecution failed: {type(e).__name__}: {e}")
    print("Possible causes: API key, rate limit, model name, network")

Connected to model: grok-4-1-fast-reasoning

Graph compiled successfully.

User query: Find me flights from London to Tokyo for August 15, 2026. 


─── Flight Search Agent ───


> Entering new AgentExecutor chain...

Invoking: `flight_search_tool` with `{'origin': 'London', 'destination': 'Tokyo', 'date': '2026-08-15'}`



--- TOOL CALLED: flight_search_tool (London → Tokyo on 2026-08-15) ---
[{'flight_number': 'BA007', 'airline': 'British Airways', 'price_gbp': 1200, 'duration_h': 13, 'layovers': 0}, {'flight_number': 'JAL404', 'airline': 'Japan Airlines', 'price_gbp': 1150, 'duration_h': 14, 'layovers': 1}]Here are available flights from London to Tokyo on 2026-08-15:

- **British Airways BA007**: £1,200, 13 hours, direct (0 layovers).
- **Japan Airlines JAL404**: £1,150, 14 hours, 1 layover.

Let me know if you'd like to review details, compare options, or book!

> Finished chain.

─── Review / Summary Agent ───

══════════════════════════════════════════════════════════════════════

## CASE2

In [3]:
# =============================================================================
#  GROK 4.1 FLIGHT AGENT – FULL CODE WITH ROUND-TRIP SUPPORT (Feb 2026)
# =============================================================================

# ─── Imports ────────────────────────────────────────────────────────────────
import os
from typing import Annotated, List, TypedDict, Optional
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langgraph.graph import StateGraph, END

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ─── Config ─────────────────────────────────────────────────────────────────
if userdata:
    XAI_API_KEY = userdata.get('XAI_KEY')
else:
    XAI_API_KEY = os.environ.get('XAI_API_KEY')

if not XAI_API_KEY:
    raise ValueError("Please set XAI_KEY in Colab secrets or environment.")

llm = ChatOpenAI(
    model          = "grok-4-1-fast-reasoning",
    temperature    = 0.2,
    api_key        = XAI_API_KEY,
    base_url       = "https://api.x.ai/v1",
    max_retries    = 2,
    request_timeout= 90,
)

print(f"✅ Connected to: {llm.model}\n")

# ─── TOOL WITH ROUND-TRIP SUPPORT ───────────────────────────────────────────
@tool
def flight_search_tool(
    origin: str,
    destination: str,
    departure_date: str,
    return_date: Optional[str] = None
) -> List[dict] | dict:
    """Flight search tool – supports one-way and round-trip (mock data)."""
    print(f"\n--- TOOL: flight_search_tool ({origin} → {destination} | {departure_date} → {return_date}) ---")

    origin = origin.lower().strip()
    destination = destination.lower().strip()

    if origin == "london" and destination == "tokyo" and departure_date == "2026-08-15":
        outbound = [
            {"type": "outbound", "flight_number": "BA007", "airline": "British Airways",   "price_gbp": 1200, "duration_h": 13, "layovers": 0},
            {"type": "outbound", "flight_number": "JAL404", "airline": "Japan Airlines",   "price_gbp": 1150, "duration_h": 14, "layovers": 1},
        ]

        if return_date:
            inbound = [
                {"type": "return", "flight_number": "BA008", "airline": "British Airways",   "price_gbp": 1350, "duration_h": 12.5, "layovers": 0},
                {"type": "return", "flight_number": "JAL405", "airline": "Japan Airlines",   "price_gbp": 1280, "duration_h": 13.5, "layovers": 1},
            ]
            total = sum(f["price_gbp"] for f in outbound + inbound)
            return outbound + inbound + [{"total_round_trip_price_gbp": total, "currency": "GBP"}]
        return outbound

    return {"message": "No matching flights found (mock tool)."}

tools = [flight_search_tool]

# ─── Prompts ────────────────────────────────────────────────────────────────
search_system_prompt = """You are a professional flight assistant.
Use the flight_search_tool tool.
Support both one-way and round-trip queries.
Always extract origin, destination, departure_date, and return_date (if mentioned)."""

search_prompt = ChatPromptTemplate.from_messages([
    ("system", search_system_prompt),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("agent_scratchpad"),
])

review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly flight summary expert.
Create a clean markdown summary.
If it's a round-trip, show the total price clearly.
Highlight the cheapest option.
Use bullet points and end with a helpful offer."""),
    MessagesPlaceholder("messages"),
])

# ─── Agents ─────────────────────────────────────────────────────────────────
flight_search_runnable = create_tool_calling_agent(
    llm,
    tools,
    search_prompt
)

flight_search_executor = AgentExecutor(
    agent=flight_search_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)

review_chain = review_prompt | llm

# ─── LangGraph ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

def run_search(state: AgentState):
    print("\n─── 🔍 Flight Search Agent ───")
    result = flight_search_executor.invoke({"messages": state["messages"]})
    return {"messages": [AIMessage(content=result.get("output", "No result"))]}

def run_review(state: AgentState):
    print("\n─── 📋 Review & Summary Agent ───")
    response = review_chain.invoke(state)
    return {"messages": [AIMessage(content=response.content)]}

workflow = StateGraph(AgentState)
workflow.add_node("search", run_search)
workflow.add_node("review", run_review)

workflow.set_entry_point("search")
workflow.add_edge("search", "review")
workflow.add_edge("review", END)

app = workflow.compile()

print("🚀 Graph compiled with round-trip support!\n")

# ─── DEMO RUNS ──────────────────────────────────────────────────────────────
queries = [
    "Find me flights from London to Tokyo on August 15, 2026.",           # one-way
    "Find me round-trip flights from London to Tokyo departing August 15, 2026 and returning August 25, 2026."  # round-trip
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*80}")
    print(f"DEMO {i}: {query}")
    print('='*80)

    initial_input = {"messages": [HumanMessage(content=query)]}

    try:
        final = app.invoke(initial_input)
        print("\n" + final["messages"][-1].content)
    except Exception as e:
        print(f"Error: {e}")

✅ Connected to: grok-4-1-fast-reasoning

🚀 Graph compiled with round-trip support!


DEMO 1: Find me flights from London to Tokyo on August 15, 2026.

─── 🔍 Flight Search Agent ───


> Entering new AgentExecutor chain...

Invoking: `flight_search_tool` with `{'origin': 'London', 'destination': 'Tokyo', 'departure_date': '2026-08-15'}`



--- TOOL: flight_search_tool (London → Tokyo | 2026-08-15 → None) ---
[{'type': 'outbound', 'flight_number': 'BA007', 'airline': 'British Airways', 'price_gbp': 1200, 'duration_h': 13, 'layovers': 0}, {'type': 'outbound', 'flight_number': 'JAL404', 'airline': 'Japan Airlines', 'price_gbp': 1150, 'duration_h': 14, 'layovers': 1}]Here are the available one-way flight options from **London (LON)** to **Tokyo (TYO)** departing on **August 15, 2026**. Prices are in GBP and based on economy class (mock data; actual availability may vary—check with airlines for real-time booking).

### Recommended Flights:
1. **British Airways (BA007)**  
   - Duration: 13 ho

## CASE3

In [5]:
# =============================================================================
#  IMPROVED GROK 4.1 FLIGHT AGENT – ROUND-TRIP + REALISTIC MOCK PRICES
# =============================================================================


import os
from typing import Annotated, List, TypedDict, Optional
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langgraph.graph import StateGraph, END

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ─── Config ─────────────────────────────────────────────────────────────────
XAI_API_KEY = userdata.get('XAI_KEY') if userdata else os.environ.get('XAI_API_KEY')
if not XAI_API_KEY:
    raise ValueError("XAI_KEY not found. Set it in Colab secrets or environment.")

llm = ChatOpenAI(
    model="grok-4-1-fast-reasoning",
    temperature=0.2,
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
    max_retries=2,
    request_timeout=90,
)

print(f"Connected to: {llm.model}\n")

# ─── TOOL ───────────────────────────────────────────────────────────────────
@tool
def flight_search_tool(
    origin: str,
    destination: str,
    departure_date: str,
    return_date: Optional[str] = None
) -> dict:
    """Mock flight search with realistic August peak-season pricing."""
    origin = origin.lower().strip()
    destination = destination.lower().strip()

    print(f"--- TOOL CALLED: {origin} → {destination} | {departure_date} ↔ {return_date} ---")

    if origin != "london" or destination != "tokyo" or departure_date != "2026-08-15":
        return {"message": "No matching flights found (mock limitation)."}

    # Realistic peak August 2026 economy prices (direct & 1-stop)
    outbound = [
        {"type": "outbound", "airline": "British Airways", "flight": "BA007", "price_gbp": 1650, "duration_h": 13, "stops": 0, "note": "direct"},
        {"type": "outbound", "airline": "Japan Airlines",   "flight": "JL042", "price_gbp": 1580, "duration_h": 14, "stops": 1},
        {"type": "outbound", "airline": "ANA",              "flight": "NH212", "price_gbp": 1720, "duration_h": 13.5, "stops": 0, "note": "direct"},
    ]

    if return_date:
        inbound = [
            {"type": "return", "airline": "British Airways", "flight": "BA008", "price_gbp": 1580, "duration_h": 12.5, "stops": 0, "note": "direct"},
            {"type": "return", "airline": "Japan Airlines",   "flight": "JL043", "price_gbp": 1520, "duration_h": 13.5, "stops": 1},
            {"type": "return", "airline": "ANA",              "flight": "NH211", "price_gbp": 1650, "duration_h": 12.8, "stops": 0, "note": "direct"},
        ]

        # Find cheapest combo
        cheapest_out = min(outbound, key=lambda x: x["price_gbp"])
        cheapest_in  = min(inbound,  key=lambda x: x["price_gbp"])
        total_cheapest = cheapest_out["price_gbp"] + cheapest_in["price_gbp"]

        return {
            "outbound": outbound,
            "return": inbound,
            "cheapest_combo": {
                "total_gbp": total_cheapest,
                "outbound": cheapest_out,
                "return": cheapest_in
            },
            "disclaimer": "Mock data – real August prices often higher due to peak summer demand"
        }

    return {"outbound": outbound, "disclaimer": "Mock data – illustrative only"}

tools = [flight_search_tool]

# ─── Prompts ────────────────────────────────────────────────────────────────
search_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional flight search assistant.
Use the flight_search_tool.
Support one-way and round-trip.
Extract origin, destination, departure_date, return_date (if any).
After results, summarize key options clearly."""),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("agent_scratchpad"),
])

review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly travel summary expert.
Summarize flight results in clean markdown.
Highlight cheapest option/combo.
Include disclaimer that these are mock/illustrative prices.
End with helpful offer."""),
    MessagesPlaceholder("messages"),
])

# ─── Agents & Graph ─────────────────────────────────────────────────────────
flight_search_runnable = create_tool_calling_agent(llm, tools, search_prompt)
executor = AgentExecutor(agent=flight_search_runnable, tools=tools, verbose=True, max_iterations=8)
#review_chain = review_prompt | llm

review_chain = review_prompt | llm.bind(max_tokens=1000, temperature=0.2)

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

def search_node(state):
    print("\n─── Flight Search ───")
    result = executor.invoke({"messages": state["messages"]})
    return {"messages": [AIMessage(content=result.get("output", ""))]}

def review_node(state):
    print("\n─── Summary ───")
    resp = review_chain.invoke(state)
    return {"messages": [AIMessage(content=resp.content)]}

workflow = StateGraph(AgentState)
workflow.add_node("search", search_node)
workflow.add_node("review", review_node)
workflow.set_entry_point("search")
workflow.add_edge("search", "review")
workflow.add_edge("review", END)

app = workflow.compile()
print("Graph ready.\n")

# ─── Demo ───────────────────────────────────────────────────────────────────
queries = [
    "Flights from London to Tokyo on August 15, 2026",
    "Round-trip London to Tokyo, depart Aug 15 2026, return Aug 25 2026"
]

for q in queries:
    print(f"\n{'═'*70}\nQuery: {q}\n{'═'*70}")
    result = app.invoke({"messages": [HumanMessage(content=q)]})
    print("\n" + result["messages"][-1].content)

Connected to: grok-4-1-fast-reasoning

Graph ready.


══════════════════════════════════════════════════════════════════════
Query: Flights from London to Tokyo on August 15, 2026
══════════════════════════════════════════════════════════════════════

─── Flight Search ───


> Entering new AgentExecutor chain...

Invoking: `flight_search_tool` with `{'origin': 'London', 'destination': 'Tokyo', 'departure_date': '2026-08-15'}`


--- TOOL CALLED: london → tokyo | 2026-08-15 ↔ None ---
{'outbound': [{'type': 'outbound', 'airline': 'British Airways', 'flight': 'BA007', 'price_gbp': 1650, 'duration_h': 13, 'stops': 0, 'note': 'direct'}, {'type': 'outbound', 'airline': 'Japan Airlines', 'flight': 'JL042', 'price_gbp': 1580, 'duration_h': 14, 'stops': 1}, {'type': 'outbound', 'airline': 'ANA', 'flight': 'NH212', 'price_gbp': 1720, 'duration_h': 13.5, 'stops': 0, 'note': 'direct'}], 'disclaimer': 'Mock data – illustrative only'}### One-Way Flights: London (LON) to Tokyo (TYO) on August 15, 202

## CASE4

In [7]:
# =============================================================================
#  GROK 4.1 FLIGHT AGENT – ROUND-TRIP + ANTI-TRUNCATION FIXES
# =============================================================================


# ─── Imports ────────────────────────────────────────────────────────────────
import os
from typing import Annotated, List, TypedDict, Optional
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langgraph.graph import StateGraph, END

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ─── Config ─────────────────────────────────────────────────────────────────
XAI_API_KEY = userdata.get('XAI_KEY') if userdata else os.environ.get('XAI_API_KEY')

if not XAI_API_KEY:
    raise ValueError("XAI_KEY not found. Set it in Colab secrets or environment.")

llm = ChatOpenAI(
    model          = "grok-4-1-fast-reasoning",
    temperature    = 0.2,
    api_key        = XAI_API_KEY,
    base_url       = "https://api.x.ai/v1",
    max_retries    = 2,
    request_timeout= 90,
)

print(f"Connected to: {llm.model}\n")

# ─── TOOL ───────────────────────────────────────────────────────────────────
@tool
def flight_search_tool(
    origin: str,
    destination: str,
    departure_date: str,
    return_date: Optional[str] = None,
    adults: int = 1,
    currency: str = "CAD"
) -> dict:
    """Mock flight search with realistic prices in CAD."""
    origin = origin.upper()[:3]
    destination = destination.upper()[:3]

    print(f"--- TOOL: {origin} → {destination} | {departure_date} ↔ {return_date} ---")

    mock_outbound = [
        {"airline": "Air Canada", "flight": "AC880", "price": 1850, "currency": "CAD", "duration": "13h 30m", "stops": 0, "note": "direct YUL-HND"},
        {"airline": "Air Transat", "flight": "TS124", "price": 1420, "currency": "CAD", "duration": "15h", "stops": 1},
        {"airline": "ANA", "flight": "NHxxx", "price": 1980, "currency": "CAD", "duration": "14h", "stops": 1},
    ]

    if return_date:
        mock_return = [
            {"airline": "Air Canada", "flight": "AC879", "price": 1720, "currency": "CAD", "duration": "12h 50m", "stops": 0},
            {"airline": "Air Transat", "flight": "TS125", "price": 1380, "currency": "CAD", "duration": "14h 30m", "stops": 1},
        ]
        cheapest_out = min(mock_outbound, key=lambda x: x["price"])
        cheapest_ret = min(mock_return,  key=lambda x: x["price"])
        total = cheapest_out["price"] + cheapest_ret["price"]
        return {
            "outbound": mock_outbound,
            "return": mock_return,
            "cheapest_combo": {"total": total, "currency": "CAD", "outbound": cheapest_out, "return": cheapest_ret},
            "source": "Mock data",
            "disclaimer": "Estimates only – peak summer prices can be higher"
        }

    return {
        "outbound": mock_outbound,
        "source": "Mock data",
        "disclaimer": "Mock prices for illustration only"
    }

tools = [flight_search_tool]

# ─── Prompts ────────────────────────────────────────────────────────────────
search_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful flight assistant for Quebec travelers.
Use flight_search_tool.
Extract origin, destination, departure_date, return_date (if mentioned), adults.
Prefer CAD currency. Summarize results clearly."""),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("agent_scratchpad"),
])

review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly Quebec travel expert.
Summarize flight results in **short, clean markdown** using CAD $.
Highlight the cheapest realistic option or combo.
Mention source (mock).
Include disclaimer: "These are estimates only – check real-time quotes".
Use bullets. End with one offer to help."""),
    MessagesPlaceholder("messages"),
])

# ─── Agents ─────────────────────────────────────────────────────────────────
flight_search_runnable = create_tool_calling_agent(llm, tools, search_prompt)

executor = AgentExecutor(
    agent=flight_search_runnable,
    tools=tools,
    verbose=True,
    max_iterations=8,
)

# Prevent truncation – this is the key fix
review_chain = review_prompt | llm.bind(max_tokens=900, temperature=0.2)

# ─── LangGraph ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

def search_node(state: AgentState):
    print("\n─── Flight Search ───")
    result = executor.invoke({"messages": state["messages"]})
    return {"messages": [AIMessage(content=result.get("output", "No result"))]}

def review_node(state: AgentState):
    print("\n─── Summary ───")
    try:
        resp = review_chain.invoke(state)
        content = resp.content
        # Small guard against obvious truncation
        if len(content) < 150 and content.strip().endswith(("...", "202", "Aug")):
            content += "\n\n*(Note: summary may have been cut short – ask for more detail if needed)*"
        return {"messages": [AIMessage(content=content)]}
    except Exception as e:
        return {"messages": [AIMessage(content=f"Summary failed: {str(e)}")]}

workflow = StateGraph(AgentState)
workflow.add_node("search", search_node)
workflow.add_node("review", review_node)
workflow.set_entry_point("search")
workflow.add_edge("search", "review")
workflow.add_edge("review", END)

app = workflow.compile()
print("Graph ready.\n")

# ─── Demo ───────────────────────────────────────────────────────────────────
queries = [
    "Flights from Montreal to Tokyo on August 15, 2026",
    "Round-trip from Quebec City to Tokyo, depart August 15 2026, return August 25 2026, for 2 adults"
]

for q in queries:
    print(f"\n{'═'*80}\nQuery: {q}\n{'═'*80}")
    try:
        result = app.invoke({"messages": [HumanMessage(content=q)]})
        print("\n" + result["messages"][-1].content)
    except Exception as e:
        print(f"Error: {e}")

Connected to: grok-4-1-fast-reasoning

Graph ready.


════════════════════════════════════════════════════════════════════════════════
Query: Flights from Montreal to Tokyo on August 15, 2026
════════════════════════════════════════════════════════════════════════════════

─── Flight Search ───


> Entering new AgentExecutor chain...

Invoking: `flight_search_tool` with `{'origin': 'Montreal', 'destination': 'Tokyo', 'departure_date': '2026-08-15', 'return_date': 'null', 'adults': 1, 'currency': 'CAD'}`


--- TOOL: MON → TOK | 2026-08-15 ↔ null ---
{'outbound': [{'airline': 'Air Canada', 'flight': 'AC880', 'price': 1850, 'currency': 'CAD', 'duration': '13h 30m', 'stops': 0, 'note': 'direct YUL-HND'}, {'airline': 'Air Transat', 'flight': 'TS124', 'price': 1420, 'currency': 'CAD', 'duration': '15h', 'stops': 1}, {'airline': 'ANA', 'flight': 'NHxxx', 'price': 1980, 'currency': 'CAD', 'duration': '14h', 'stops': 1}], 'return': [{'airline': 'Air Canada', 'flight': 'AC879', 'price': 1720, 'c

## CASE5

In [8]:
# =============================================================================
#  FINAL GROK 4.1 FLIGHT AGENT – QUEBEC EDITION (Fully Polished)
#  Montreal / Quebec City focus • CAD prices • No truncation • Round-trip
# =============================================================================


# ─── Imports ────────────────────────────────────────────────────────────────
import os
from typing import Annotated, List, TypedDict, Optional
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langgraph.graph import StateGraph, END

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ─── Config ─────────────────────────────────────────────────────────────────
XAI_API_KEY = userdata.get('XAI_KEY') if userdata else os.environ.get('XAI_API_KEY')

if not XAI_API_KEY:
    raise ValueError("Please set XAI_KEY in Colab secrets (left sidebar → key icon)")

llm = ChatOpenAI(
    model="grok-4-1-fast-reasoning",
    temperature=0.2,
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
    max_retries=2,
    request_timeout=90,
)

print(f"✅ Connected to Grok-4-1-fast-reasoning\n")

# ─── TOOL (Realistic CAD mock for Quebec travelers) ─────────────────────────
@tool
def flight_search_tool(
    origin: str,
    destination: str,
    departure_date: str,
    return_date: Optional[str] = None,
    adults: int = 1,
    currency: str = "CAD"
) -> dict:
    """Flight search – realistic mock data for Quebec → Tokyo (peak August)."""
    origin = origin.upper()[:3]
    destination = destination.upper()[:3]

    print(f"--- TOOL: {origin} → {destination} | {departure_date} ↔ {return_date} ({adults} adults) ---")

    outbound = [
        {"airline": "Air Canada", "flight": "AC880", "price": 1850, "currency": "CAD", "duration": "13h 30m", "stops": 0, "note": "YUL-HND direct"},
        {"airline": "Air Transat", "flight": "TS124", "price": 1420, "currency": "CAD", "duration": "15h", "stops": 1},
        {"airline": "ANA", "flight": "NHxxx", "price": 1980, "currency": "CAD", "duration": "14h", "stops": 1},
    ]

    if return_date:
        return_flights = [
            {"airline": "Air Canada", "flight": "AC879", "price": 1720, "currency": "CAD", "duration": "12h 50m", "stops": 0},
            {"airline": "Air Transat", "flight": "TS125", "price": 1380, "currency": "CAD", "duration": "14h 30m", "stops": 1},
        ]
        cheapest_out = min(outbound, key=lambda x: x["price"])
        cheapest_ret = min(return_flights, key=lambda x: x["price"])
        total_per_person = cheapest_out["price"] + cheapest_ret["price"]
        total_all = total_per_person * adults

        return {
            "outbound": outbound,
            "return": return_flights,
            "cheapest_combo": {
                "total_per_person": total_per_person,
                "total_for_all": total_all,
                "currency": "CAD",
                "outbound": cheapest_out,
                "return": cheapest_ret
            },
            "source": "Mock data (realistic peak August 2026)",
            "disclaimer": "Estimates only – actual prices vary. Book early for best rates."
        }

    return {
        "outbound": outbound,
        "source": "Mock data",
        "disclaimer": "Estimates only – actual prices vary."
    }

tools = [flight_search_tool]

# ─── Prompts ────────────────────────────────────────────────────────────────
search_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful flight assistant for travelers from Quebec.
Use the flight_search_tool.
Extract origin, destination, departure_date, return_date (if any), adults.
Prefer CAD currency and mention connections via YUL when relevant.
Summarize clearly after getting results."""),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("agent_scratchpad"),
])

review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly Quebec travel expert.
Summarize in short, clean markdown using CAD $.
Show airport codes (YUL, YQB, NRT/HND etc.).
Highlight cheapest combo and total for all passengers.
Mention connections via Montreal if relevant.
Include disclaimer.
Use bullets/tables. End with one helpful offer."""),
    MessagesPlaceholder("messages"),
])

# ─── Agents ─────────────────────────────────────────────────────────────────
flight_search_runnable = create_tool_calling_agent(llm, tools, search_prompt)

executor = AgentExecutor(agent=flight_search_runnable, tools=tools, verbose=True, max_iterations=8)

# Anti-truncation fix
review_chain = review_prompt | llm.bind(max_tokens=950, temperature=0.2)

# ─── LangGraph ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

def search_node(state: AgentState):
    print("\n─── 🔍 Flight Search Agent ───")
    result = executor.invoke({"messages": state["messages"]})
    return {"messages": [AIMessage(content=result.get("output", "No result"))]}

def review_node(state: AgentState):
    print("\n─── 📋 Summary Agent ───")
    resp = review_chain.invoke(state)
    content = resp.content
    if len(content) < 200 and content.strip().endswith(("202", "Aug", "...")):
        content += "\n\n*(Summary complete – ask for more details if needed)*"
    return {"messages": [AIMessage(content=content)]}

workflow = StateGraph(AgentState)
workflow.add_node("search", search_node)
workflow.add_node("review", review_node)
workflow.set_entry_point("search")
workflow.add_edge("search", "review")
workflow.add_edge("review", END)

app = workflow.compile()
print("🚀 Graph ready – Quebec Edition!\n")

# ─── RUN DEMOS ──────────────────────────────────────────────────────────────
queries = [
    "Flights from Montreal to Tokyo on August 15, 2026",
    "Round-trip from Quebec City to Tokyo, depart August 15 2026, return August 25 2026, for 2 adults"
]

for q in queries:
    print(f"\n{'═'*85}\nQuery → {q}\n{'═'*85}")
    result = app.invoke({"messages": [HumanMessage(content=q)]})
    print("\n" + result["messages"][-1].content)

✅ Connected to Grok-4-1-fast-reasoning

🚀 Graph ready – Quebec Edition!


═════════════════════════════════════════════════════════════════════════════════════
Query → Flights from Montreal to Tokyo on August 15, 2026
═════════════════════════════════════════════════════════════════════════════════════

─── 🔍 Flight Search Agent ───


> Entering new AgentExecutor chain...

Invoking: `flight_search_tool` with `{'origin': 'Montreal', 'destination': 'Tokyo', 'departure_date': '2026-08-15', 'return_date': 'null', 'adults': 1}`


--- TOOL: MON → TOK | 2026-08-15 ↔ null (1 adults) ---
{'outbound': [{'airline': 'Air Canada', 'flight': 'AC880', 'price': 1850, 'currency': 'CAD', 'duration': '13h 30m', 'stops': 0, 'note': 'YUL-HND direct'}, {'airline': 'Air Transat', 'flight': 'TS124', 'price': 1420, 'currency': 'CAD', 'duration': '15h', 'stops': 1}, {'airline': 'ANA', 'flight': 'NHxxx', 'price': 1980, 'currency': 'CAD', 'duration': '14h', 'stops': 1}], 'return': [{'airline': 'Air Canada', 'flig